# 06 — Preprocessing and Baseline Feature Design

## Purpose

Notebook 05 selected the **5-minute SOLETE data** as the primary modelling
resolution. We now define the transformations required before forecasting.

The goal is deliberately narrow:

1. correct source-level artifacts only when the evidence is strong;
2. preserve raw observations for traceability;
3. represent circular quantities appropriately;
4. distinguish source-quality control from target modification;
5. define what information is available at forecast issue time versus
   forecast-valid time; and
6. prepare a common baseline that can support both **PV** and **wind**
   forecasting without leaking future measurements.

No model is trained here and no scaler is fitted here.

## What earlier SOLETE research contributes

Prior SOLETE studies are useful methodological references.

- **Pombo, Gehrke & Bindner (2022), Data in Brief,
  DOI 10.1016/j.dib.2022.108046** documents the co-located meteorological,
  10 kW PV, and 11 kW wind measurements and the original aggregation to
  5-minute and hourly data.
- **Pombo et al. (2022), Energy Reports,
  DOI 10.1016/j.egyr.2022.05.006** used 5-minute SOLETE for short-term PV
  forecasting and showed the value of physically informed predictors.
- **Pombo et al. (2022), Sustainable Energy, Grids and Networks,
  DOI 10.1016/j.segan.2022.100943** forecast both co-located PV and wind and
  investigated wind-specific quantities such as air density, turbulence
  intensity, and an effective turbine power curve.
- The later **SOLETE platform v3+** corrected previously documented
  train/validation/test splitting and RMSE-processing bugs. We therefore
  reconstruct the forecasting protocol independently rather than inheriting
  old platform splits.
- **Yang & Niu (2025), DOI 10.1016/j.seta.2025.104572** provides more recent
  evidence that SOLETE remains useful for both PV and wind forecasting across
  multiple temporal segmentations.

Some later studies apply generic interpolation, smoothing, or outlier
replacement. We do **not** adopt those operations automatically: our direct
audit found a complete 5-minute grid, and abrupt renewable-power changes can
be physically real. Cleaning below is therefore artifact-specific and
evidence-driven.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
DATA_1MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_1min.h5"

for path in [DATA_5MIN, DATA_1MIN]:
    if not path.exists():
        raise FileNotFoundError(path)

df_raw = load_solete(DATA_5MIN).sort_index()
df_work = df_raw.copy(deep=True)

# These boundaries are used only to keep source-quality rule development
# separate from the final test period. Forecast validation will later use
# time-series folds inside the development period.
TRAIN_END = pd.Timestamp("2019-04-01", tz="UTC")
TEST_START = pd.Timestamp("2019-06-01", tz="UTC")
TEST_END = pd.Timestamp("2019-09-01", tz="UTC")

midnight = (
    (df_raw.index.hour == 0)
    & (df_raw.index.minute == 0)
    & (df_raw.index.second == 0)
)

print("5-minute dataset:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Chronological:", df_raw.index.is_monotonic_increasing)
print("Duplicate timestamps:", int(df_raw.index.duplicated().sum()))
print("Missing measurements:", int(df_raw.isna().sum().sum()))


5-minute dataset: (131617, 11)
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Chronological: True
Duplicate timestamps: 0
Missing measurements: 0


## 1. Source-quality register

The earlier EDA identified three repeatable measurement/representation issues:

- atmospheric pressure contains a daily-boundary high cluster;
- humidity is numerically fractional despite its source label and contains
  isolated values above 1 at the daily boundary;
- Wind direction occasionally lies outside the conventional $[0^\circ, 360^\circ)$ interval.

These are treated separately from the Gaia wind-power question. Power values
are targets, so they should not be overwritten merely because they look
unusual.


In [2]:
pressure_raw = df_raw["Pressure[mbar]"]
humidity_raw = df_raw["HUMIDITY[%]"]
wind_dir_raw = df_raw["WIND_DIR[deg]"]

pressure_flag = pressure_raw > 1100
humidity_flag = humidity_raw > 1
wind_dir_flag = (wind_dir_raw < 0) | (wind_dir_raw >= 360)

issue_register = pd.DataFrame(
    {
        "observations": [
            int(pressure_flag.sum()),
            int(humidity_flag.sum()),
            int(wind_dir_flag.sum()),
        ],
        "all_at_00_00_UTC": [
            bool((~pressure_flag | midnight).all()),
            bool((~humidity_flag | midnight).all()),
            "not required",
        ],
        "planned_action": [
            "dataset-specific pressure correction",
            "causal replacement from previous valid value",
            "wrap modulo 360 and encode circularly",
        ],
    },
    index=[
        "Pressure > 1100 mbar",
        "Humidity > 1",
        "Wind direction outside [0,360)",
    ],
)

display(issue_register)


,observations,all_at_00_00_UTC,planned_action
Pressure > 1100 mbar,456,True,dataset-specific pressure correction
Humidity > 1,164,True,causal replacement from previous valid value
"Wind direction outside [0,360)",103,not required,wrap modulo 360 and encode circularly


## 2. Pressure: diagnose before correcting

The pressure values above 1100 mbar are not treated as impossible merely
because of a generic atmospheric threshold. The important evidence is
dataset-specific:

- they form a separate high cluster;
- they occur at the same daily boundary; and
- they are approximately **1000 mbar above** the neighbouring pressure
  trajectory.

For diagnosis only, the average of the observations immediately before and
after an affected timestamp provides an offline local reference. That
two-sided reference is **not** used as a model feature.

We compare two causal repair candidates:

1. subtract 1000 mbar from the affected source value;
2. copy the preceding 5-minute pressure observation.


In [3]:
pressure_prev = pressure_raw.shift(1)
pressure_next = pressure_raw.shift(-1)
pressure_local_reference = (pressure_prev + pressure_next) / 2

pressure_minus_1000 = pressure_raw.where(
    ~pressure_flag,
    pressure_raw - 1000,
)
pressure_previous = pressure_raw.where(
    ~pressure_flag,
    pressure_prev,
)

pressure_diagnostic = pd.Series(
    {
        "flagged_observations": int(pressure_flag.sum()),
        "median_raw_jump_from_previous_mbar":
            float((pressure_raw - pressure_prev).loc[pressure_flag].median()),
        "MAE_subtract_1000_vs_local_reference_mbar":
            float(
                (
                    pressure_minus_1000.loc[pressure_flag]
                    - pressure_local_reference.loc[pressure_flag]
                ).abs().mean()
            ),
        "MAE_previous_value_vs_local_reference_mbar":
            float(
                (
                    pressure_previous.loc[pressure_flag]
                    - pressure_local_reference.loc[pressure_flag]
                ).abs().mean()
            ),
    },
    name="pressure diagnostic",
)

display(pressure_diagnostic.to_frame("value"))

# Final rule: preserve the source column and create a cleaned modelling column.
df_work["pressure_anomaly_flag"] = pressure_flag.astype("int8")
df_work["Pressure_clean[mbar]"] = pressure_minus_1000

assert df_work["Pressure_clean[mbar]"].notna().all()
assert df_work.loc[pressure_flag, "Pressure_clean[mbar]"].max() < 1100


,value
flagged_observations,456.000000
median_raw_jump_from_previous_mbar,1000.000000
MAE_subtract_1000_vs_local_reference_mbar,0.111836
MAE_previous_value_vs_local_reference_mbar,0.114658


### Pressure decision

Subtracting 1000 mbar preserves the information in the affected timestamp and
matches its local pressure trajectory slightly better than simply copying the
previous observation. The correction is therefore retained as a
**SOLETE-specific artifact rule**, not as a universal pressure-cleaning rule.


## 3. Humidity: preserve causality

Most humidity values behave as fractions in $[0,1]$, despite the source
column name containing `%`. Values above 1 occur only at the same daily
boundary.

We compare the hypothesis that these values contain a simple `+1` offset with
a causal replacement based on the most recent valid observation.

As with pressure, a two-sided neighbour average is used only to judge the
candidate rules offline. The final transformation itself never uses a future
measurement.


In [4]:
humidity_prev = humidity_raw.shift(1)
humidity_next = humidity_raw.shift(-1)
humidity_local_reference = (humidity_prev + humidity_next) / 2

humidity_minus_1 = humidity_raw.where(~humidity_flag, humidity_raw - 1)

# Remove the invalid records, then carry only past valid information forward.
humidity_causal = humidity_raw.mask(humidity_flag).ffill()

humidity_diagnostic = pd.Series(
    {
        "flagged_observations": int(humidity_flag.sum()),
        "MAE_subtract_1_vs_local_reference":
            float(
                (
                    humidity_minus_1.loc[humidity_flag]
                    - humidity_local_reference.loc[humidity_flag]
                ).abs().mean()
            ),
        "MAE_causal_previous_valid_vs_local_reference":
            float(
                (
                    humidity_causal.loc[humidity_flag]
                    - humidity_local_reference.loc[humidity_flag]
                ).abs().mean()
            ),
        "median_causal_error":
            float(
                (
                    humidity_causal.loc[humidity_flag]
                    - humidity_local_reference.loc[humidity_flag]
                ).abs().median()
            ),
    },
    name="humidity diagnostic",
)

display(humidity_diagnostic.to_frame("value"))

df_work["humidity_anomaly_flag"] = humidity_flag.astype("int8")
df_work["Humidity_clean"] = humidity_causal

assert df_work["Humidity_clean"].notna().all()
assert df_work["Humidity_clean"].between(0, 1).all()


,value
flagged_observations,164.000000
MAE_subtract_1_vs_local_reference,0.153634
MAE_causal_previous_valid_vs_local_reference,0.002198
median_causal_error,0.000000


### Humidity decision

The `+1` hypothesis does not reproduce the local humidity trajectory nearly as
well as the preceding valid measurement. The 164 affected values are therefore
treated as isolated source artifacts and replaced **causally**.

The clean modelling series remains on a fractional scale. We do not multiply
it by 100 merely because the original column label contains `%`.


## 4. Wind direction: change the representation, not the observation

Wind direction is circular. Numerically, $359^\circ$ and $1^\circ$ are far
apart, although physically they are adjacent.

The raw source value remains available for traceability. The modelling
representation is

$$
\theta_w = \theta \bmod 360^\circ,
$$

followed by

$$
\left[
\sin\left(\frac{\pi\theta_w}{180}\right),
\cos\left(\frac{\pi\theta_w}{180}\right)
\right].
$$

In [5]:
wind_dir_wrapped = wind_dir_raw.mod(360)
wind_radians = np.deg2rad(wind_dir_wrapped)

df_work["wind_dir_wrap_flag"] = wind_dir_flag.astype("int8")
df_work["WIND_DIR_wrapped[deg]"] = wind_dir_wrapped
df_work["wind_dir_sin"] = np.sin(wind_radians)
df_work["wind_dir_cos"] = np.cos(wind_radians)

wind_norm_error = np.abs(
    df_work["wind_dir_sin"] ** 2
    + df_work["wind_dir_cos"] ** 2
    - 1
).max()

display(
    pd.Series(
        {
            "raw_out_of_range_count": int(wind_dir_flag.sum()),
            "wrapped_min_deg": float(wind_dir_wrapped.min()),
            "wrapped_max_deg": float(wind_dir_wrapped.max()),
            "max_sin2_plus_cos2_error": float(wind_norm_error),
        },
        name="wind-direction verification",
    ).to_frame("value")
)


,value
raw_out_of_range_count,1.030000e+02
wrapped_min_deg,0.000000e+00
wrapped_max_deg,3.566388e+02
max_sin2_plus_cos2_error,2.220446e-16


## 5. Gaia wind-power quality: flag, do not impute

`P_Gaia[kW]` is a candidate forecasting target. Therefore a generic outlier
filter would be risky: real wind-power ramps and high-output events should not
be smoothed away.

Notebook 05 established that the 1-minute and 5-minute files are numerically
very close under 1-minute → 5-minute aggregation. We use that relationship
only as an **offline source-integrity reference**.

To avoid choosing a quality threshold from the final test period, the
cross-resolution rule is frozen using the pre-test development period
(`timestamp < 2019-06-01`).

A round 3 kW discrepancy threshold is accepted only if **all non-midnight
development observations** remain below it. The threshold is not optimized
against forecasting accuracy and the 1-minute reference will never enter a
model feature vector.


In [6]:
df_1min = load_solete(DATA_1MIN).sort_index()

gaia_1min_to_5min = (
    df_1min["P_Gaia[kW]"]
    .resample("5min")
    .mean()
    .reindex(df_raw.index)
)

gaia_xres_error = (
    df_raw["P_Gaia[kW]"] - gaia_1min_to_5min
).abs()

development = df_raw.index < TEST_START
final_test = (df_raw.index >= TEST_START) & (df_raw.index < TEST_END)

GAIA_XRES_THRESHOLD_KW = 3.0

dev_nonmidnight_error = gaia_xres_error.loc[development & ~midnight].dropna()
dev_midnight_error = gaia_xres_error.loc[development & midnight].dropna()

print("Maximum non-midnight cross-resolution error in development:")
print(f"{dev_nonmidnight_error.max():.6f} kW")

assert dev_nonmidnight_error.max() < GAIA_XRES_THRESHOLD_KW, (
    "The pre-specified 3 kW separator no longer lies above every "
    "non-midnight development discrepancy."
)

gaia_quality_flag = midnight & (gaia_xres_error > GAIA_XRES_THRESHOLD_KW)
df_work["gaia_target_quality_flag"] = gaia_quality_flag.astype("int8")

gaia_quality_summary = pd.Series(
    {
        "threshold_kW": GAIA_XRES_THRESHOLD_KW,
        "development_flags": int((gaia_quality_flag & development).sum()),
        "final_test_flags_after_rule_was_frozen":
            int((gaia_quality_flag & final_test).sum()),
        "total_flags": int(gaia_quality_flag.sum()),
        "dataset_percent_flagged": float(100 * gaia_quality_flag.mean()),
        "largest_nonmidnight_development_error_kW":
            float(dev_nonmidnight_error.max()),
        "median_midnight_development_error_kW":
            float(dev_midnight_error.median()),
    },
    name="Gaia source-quality rule",
)

display(gaia_quality_summary.to_frame("value"))

# The wind-power measurements themselves are never overwritten.
assert df_work["P_Gaia[kW]"].equals(df_raw["P_Gaia[kW]"])


Maximum non-midnight cross-resolution error in development:
2.656057 kW


,value
threshold_kW,3.000000
development_flags,45.000000
final_test_flags_after_rule_was_frozen,3.000000
total_flags,48.000000
dataset_percent_flagged,0.036469
largest_nonmidnight_development_error_kW,2.656057
median_midnight_development_error_kW,0.000000


### How the Gaia flag will be used

The flag is metadata, not a predictor.

For a **wind forecasting** sample to enter the primary quality-screened
analysis:

1. its future `P_Gaia` target timestamp must not be flagged; and
2. none of the `P_Gaia` timestamps used in its autoregressive history may be
   flagged.

This matters because removing only corrupted targets would still allow a
questionable midnight Gaia observation to enter a later sample as an input.

We will also retain an **all-raw wind sensitivity analysis** so that the
conclusion does not depend silently on removing these few observations.

For the primary PV task, Gaia-quality flags have no effect because historical
Gaia power is not part of the default PV F0 block.


## 6. Forecast-time availability and leakage control

The forecasting sample has two times:

$$
t=\text{issue time},
\qquad
v=t+h=\text{forecast-valid time}.
$$

Measured weather, irradiance, and generation can be used only from $t$ or
earlier unless an explicit external forecast product is introduced.

Calendar and astronomical variables are different: they are deterministic
functions of timestamp (and location for astronomy), so they can be evaluated
at $v$ without observing the future weather or power.

We also postpone scaling. Any scaler required by a model must be fitted **only
on that model's training fold** and then applied to its validation/test data.

In [7]:
availability_register = pd.DataFrame(
    [
        ("TEMPERATURE[degC]", "measurement", "history only"),
        ("Humidity_clean", "cleaned measurement", "history only"),
        ("WIND_SPEED[m1s]", "measurement", "history only"),
        ("wind_dir_sin/cos", "derived from measured wind direction", "history only"),
        ("Pressure_clean[mbar]", "cleaned measurement", "history only"),
        ("GHI[kW1m2]", "measurement", "history only"),
        ("POA Irr[kW1m2]", "measurement", "history only"),
        ("P_Solar[kW]", "candidate target", "history + target at valid time"),
        ("P_Gaia[kW]", "candidate target", "history + target at valid time"),
        ("source Azimuth/Elevation", "source diagnostic", "exclude from model"),
        ("Gregorian features", "deterministic timestamp features", "valid time allowed"),
        ("independent astronomy", "deterministic time/location features", "valid time allowed"),
        ("quality-control flags", "offline metadata", "never a predictor"),
    ],
    columns=["variable_or_group", "type", "forecast_use"],
)

display(availability_register)


,variable_or_group,type,forecast_use
0,TEMPERATURE[degC],measurement,history only
1,Humidity_clean,cleaned measurement,history only
2,WIND_SPEED[m1s],measurement,history only
3,wind_dir_sin/cos,derived from measured wind direction,history only
4,Pressure_clean[mbar],cleaned measurement,history only
5,GHI[kW1m2],measurement,history only
6,POA Irr[kW1m2],measurement,history only
7,P_Solar[kW],candidate target,history + target at valid time
8,P_Gaia[kW],candidate target,history + target at valid time
9,source Azimuth/Elevation,source diagnostic,exclude from model


## 7. F1 Gregorian representation: compute once, select at valid time

To prevent accidental issue-time alignment, Gregorian features are stored in a
**separate timestamp-indexed table** rather than being mixed into the cleaned
measurement table.

F1 is fixed as **15 deterministic UTC variables**:

- 9 direct variables: year, month, day, weekday, day of year, hour, minute,
  minute of day, weekend indicator;
- 6 cyclical variables: daily, weekly, and annual sine/cosine pairs.

During sample construction for horizon $h$, the row at
`valid_time = issue_time + h` is selected from this table.

In [8]:
idx = df_raw.index

gregorian = pd.DataFrame(index=idx)

# Nine direct UTC variables.
gregorian["utc_year"] = idx.year
gregorian["utc_month"] = idx.month
gregorian["utc_day"] = idx.day
gregorian["utc_day_of_week"] = idx.dayofweek
gregorian["utc_day_of_year"] = idx.dayofyear
gregorian["utc_hour"] = idx.hour
gregorian["utc_minute"] = idx.minute
gregorian["utc_minute_of_day"] = idx.hour * 60 + idx.minute
gregorian["utc_is_weekend"] = (idx.dayofweek >= 5).astype("int8")

# Six cyclical variables.
daily_phase = 2 * np.pi * gregorian["utc_minute_of_day"] / 1440.0
weekly_phase = 2 * np.pi * gregorian["utc_day_of_week"] / 7.0

days_in_year = np.where(idx.is_leap_year, 366.0, 365.0)
annual_position = (
    (gregorian["utc_day_of_year"] - 1)
    + gregorian["utc_minute_of_day"] / 1440.0
)
annual_phase = 2 * np.pi * annual_position / days_in_year

gregorian["utc_time_sin"] = np.sin(daily_phase)
gregorian["utc_time_cos"] = np.cos(daily_phase)
gregorian["utc_weekday_sin"] = np.sin(weekly_phase)
gregorian["utc_weekday_cos"] = np.cos(weekly_phase)
gregorian["utc_year_sin"] = np.sin(annual_phase)
gregorian["utc_year_cos"] = np.cos(annual_phase)

assert gregorian.shape[1] == 15
assert not gregorian.isna().any().any()

print("Gregorian F1 variables:", gregorian.shape[1])
display(gregorian.head())


Gregorian F1 variables: 15


,utc_year,utc_month,utc_day,utc_day_of_week,utc_day_of_year,utc_hour,utc_minute,utc_minute_of_day,utc_is_weekend,utc_time_sin,utc_time_cos,utc_weekday_sin,utc_weekday_cos,utc_year_sin,utc_year_cos
Timestamp,,,,,,,,,,,,,,,
2018-06-01 00:00:00+00:00,2018,6,1,4,152,0,0,0,0,0.000000,1.000000,-0.433884,-0.900969,0.516062,-0.856551
2018-06-01 00:05:00+00:00,2018,6,1,4,152,0,5,5,0,0.021815,0.999762,-0.433884,-0.900969,0.516011,-0.856582
2018-06-01 00:10:00+00:00,2018,6,1,4,152,0,10,10,0,0.043619,0.999048,-0.433884,-0.900969,0.515960,-0.856613
2018-06-01 00:15:00+00:00,2018,6,1,4,152,0,15,15,0,0.065403,0.997859,-0.433884,-0.900969,0.515909,-0.856644
2018-06-01 00:20:00+00:00,2018,6,1,4,152,0,20,20,0,0.087156,0.996195,-0.433884,-0.900969,0.515858,-0.856674


## 8. Target-specific F0 without changing the representation question

For the main comparison, PV and wind should be treated as **parallel
single-target forecasting tasks**, not as one multi-output model.

The common historical channels are kept as similar as possible. The power
channel is target-specific:

- PV task → historical `P_Solar[kW]`;
- wind task → historical `P_Gaia[kW]`.

A useful common nine-channel template is:

1. target power;
2. GHI;
3. POA irradiance;
4. ambient temperature;
5. cleaned fractional humidity;
6. cleaned pressure;
7. wind speed;
8. sine of wind direction;
9. cosine of wind direction.

At 5-minute resolution, the established baseline uses the current issue-time
observation plus the previous 12 positions (one hour of history), together
with the target's previous-day lag. The exact sample builder will enforce this
causally.

Wind-specific physics-informed features such as **air density, turbulence
intensity, and an effective turbine power curve** are scientifically
interesting because prior SOLETE work found them useful. They should be tested
later as a **separate wind sensitivity block**, not inserted into F0 silently,
otherwise the main F0→F4 representation comparison would no longer be
symmetric and interpretable.


In [9]:
common_history = [
    "GHI[kW1m2]",
    "POA Irr[kW1m2]",
    "TEMPERATURE[degC]",
    "Humidity_clean",
    "Pressure_clean[mbar]",
    "WIND_SPEED[m1s]",
    "wind_dir_sin",
    "wind_dir_cos",
]

target_design = pd.DataFrame(
    [
        {
            "task": "PV",
            "target": "P_Solar[kW]",
            "target_history": "P_Solar[kW]",
            "previous_day_lag": "P_Solar[kW] at t-24h",
            "Gaia_quality_mask": "not required in primary F0",
        },
        {
            "task": "Wind",
            "target": "P_Gaia[kW]",
            "target_history": "P_Gaia[kW]",
            "previous_day_lag": "P_Gaia[kW] at t-24h",
            "Gaia_quality_mask":
                "target and required target-history timestamps must be clean",
        },
    ]
)

feature_family_plan = pd.DataFrame(
    [
        ("F0", "historical observations + target autoregression", "issue time/history"),
        ("F1", "F0 + 15 Gregorian UTC features", "forecast-valid time"),
        ("F2", "F1 + independently computed solar geometry", "forecast-valid time"),
        ("F3", "F2 + continuous lunar/Sun–Moon geometry", "forecast-valid time"),
        ("F4", "F3 + categorical Panchang representation", "forecast-valid time"),
    ],
    columns=["family", "definition", "deterministic_alignment"],
)

display(target_design)
display(feature_family_plan)


,task,target,target_history,previous_day_lag,Gaia_quality_mask
0,PV,P_Solar[kW],P_Solar[kW],P_Solar[kW] at t-24h,not required in primary F0
1,Wind,P_Gaia[kW],P_Gaia[kW],P_Gaia[kW] at t-24h,target and required target-history timestamps ...


,family,definition,deterministic_alignment
0,F0,historical observations + target autoregression,issue time/history
1,F1,F0 + 15 Gregorian UTC features,forecast-valid time
2,F2,F1 + independently computed solar geometry,forecast-valid time
3,F3,F2 + continuous lunar/Sun–Moon geometry,forecast-valid time
4,F4,F3 + categorical Panchang representation,forecast-valid time


## 9. Final preprocessing protocol

The core 5-minute preprocessing decisions are now frozen before forecasting:

| Source issue | Final treatment |
|---|---|
| pressure daily-boundary artifact | subtract 1000 mbar only for the confirmed high-cluster records |
| humidity values above 1 | mark invalid and causally carry the previous valid fractional value forward |
| wind direction outside conventional range | wrap modulo 360 and use sine/cosine for modelling |
| questionable Gaia midnight records | retain raw values, attach cross-resolution quality flag |
| source azimuth/elevation | retain for diagnostics only; compute solar geometry independently later |
| scaling | not performed here; fit inside each training fold only |
| future measured weather/irradiance | never treated as known future information |

The original source columns remain unchanged:


In [10]:
original_columns_unchanged = all(
    df_work[col].equals(df_raw[col])
    for col in df_raw.columns
)

verification = pd.Series(
    {
        "rows": len(df_work),
        "missing_values_in_clean_pressure":
            int(df_work["Pressure_clean[mbar]"].isna().sum()),
        "missing_values_in_clean_humidity":
            int(df_work["Humidity_clean"].isna().sum()),
        "pressure_flags": int(df_work["pressure_anomaly_flag"].sum()),
        "humidity_flags": int(df_work["humidity_anomaly_flag"].sum()),
        "wind_direction_wrap_flags": int(df_work["wind_dir_wrap_flag"].sum()),
        "Gaia_quality_flags": int(df_work["gaia_target_quality_flag"].sum()),
        "original_source_columns_unchanged": original_columns_unchanged,
    },
    name="preprocessing verification",
)

display(verification.to_frame("value"))

assert original_columns_unchanged


,value
rows,131617
missing_values_in_clean_pressure,0
missing_values_in_clean_humidity,0
pressure_flags,456
humidity_flags,164
wind_direction_wrap_flags,103
Gaia_quality_flags,48
original_source_columns_unchanged,True


## Next step

The cleaned observational foundation is now separated from deterministic
future-known representations.

The next stage should validate and construct **independent solar astronomy**
rather than trusting the source `Azimuth[deg]` and `Elevation[deg]` columns
blindly. That validation will also determine the SOLETE timestamp reference
convention before F2, F3, and F4 are finalized.

Later forecasting notebooks must additionally enforce:

- valid-time split assignment;
- rolling/blocked temporal validation rather than shuffled K-fold CV;
- training-fold-only scaling/tuning;
- target-specific Gaia quality masks for wind;
- and an all-raw wind sensitivity analysis.

This keeps source cleaning, temporal alignment, feature representation, and
model evaluation as separate auditable decisions.
